# Setup

In [1]:
import os
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()
api_key = os.environ["OPENAI_API_KEY"]

client = OpenAI(
    api_key=api_key,
)  # OPENAI_API_KEY 환경변수 사용

def llm(prompt, stop=["\n"]):
    resp = client.chat.completions.create(
        model="gpt-4.1-mini",              # 여기 모델만 바꿔서 쓰면 됨
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=1000,
        stop=stop,
    )
    return resp.choices[0].message.content

In [2]:
import wikienv, wrappers
import requests
env = wikienv.WikiEnv()
env = wrappers.HotPotQAWrapper(env, split="dev")
env = wrappers.LoggingWrapper(env)

def step(env, action):
    attempts = 0
    while attempts < 10:
        try:
            return env.step(action)
        except requests.exceptions.Timeout:
            attempts += 1

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


# ReAct

In [3]:
import json
import sys

folder = './prompts/'
prompt_file = 'prompts_naive.json'
with open(folder + prompt_file, 'r') as f:
    prompt_dict = json.load(f)

webthink_examples = prompt_dict['webthink_simple6']
# instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types:
# (1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
# (2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
# (3) Finish[answer], which returns the answer and finishes the task.
# Here are some examples.
# """
instruction = """Thought, Action, Observation 단계를 번갈아가며 수행하여 질문 응답 과제를 해결하시오. 모든 수행은 한국어로 진행하시오.

Thought는 현재 상황에 대해 추론하는 단계이며,
Action은 다음 세 가지 유형 중 하나여야 한다:

(1) Search[entity]
    Wikipedia에서 해당 entity를 정확히 검색하고, 존재할 경우 첫 번째 문단을 반환한다.
    존재하지 않을 경우, 검색 가능한 유사한 entity 목록을 반환한다.

(2) Lookup[keyword]
    현재 문서에서 해당 keyword가 포함된 다음 문장을 반환한다.

(3) Finish[answer]
    최종 답을 반환하고 과제를 종료한다.

다음은 몇 가지 예시이다.
"""
webthink_prompt = instruction + webthink_examples

def webthink(idx=None, prompt=webthink_prompt, to_print=True):
    question = env.reset(idx=idx)
    if to_print:
        print(idx, question)
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0
    for i in range(1, 8):
        n_calls += 1
        thought_action = llm(prompt + f"Thought {i}:", stop=[f"\nObservation {i}:"])
        try:
            thought, action = thought_action.strip().split(f"\nAction {i}: ")
        except:
            print('ohh...', thought_action)
            n_badcalls += 1
            n_calls += 1
            thought = thought_action.strip().split('\n')[0]
            action = llm(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=[f"\n"]).strip()
        obs, r, done, info = step(env, action[0].lower() + action[1:])
        obs = obs.replace('\\n', '')
        step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
        prompt += step_str
        if to_print:
            print(step_str)
        if done:
            break
    if not done:
        obs, r, done, info = step(env, "finish[]")
    if to_print:
        print(info, '\n')
    info.update({'n_calls': n_calls, 'n_badcalls': n_badcalls, 'traj': prompt})
    return r, info

In [4]:
import random
import time
idxs = list(range(7405))
random.Random(233).shuffle(idxs)

rs = []
infos = []
old_time = time.time()
for i in idxs[:500]:
    r, info = webthink(i, to_print=True)
    rs.append(info['em'])
    infos.append(info)
    print(sum(rs), len(rs), sum(rs) / len(rs), (time.time() - old_time) / len(rs))
    print('-----------')
    print()

3687 Question: What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?
resp text: <!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-la
Thought 1: Thought 1: 질문은 "배우 아이린 제이콥이 스튜어트 버드가 감독한 미국 액션 범죄 스릴러 영화 전에 완성한 영화가 무엇인가"이다. 스튜어트 버드가 감독한 미국 액션 범죄 스릴러 영화를 찾아야 한다. 그 후 아이린 제이콥이 그 영화 전에 완성한 영화를 찾아야 한다.

Action 1: Search[Stuart Bird]
Observation 1: Could not find Stuart Bird. Similar: ['State Bird Provisions', 'Stuart Little', 'The Good Lord Bird (miniseries)', 'Stuart Keith', 'Flappy Bird'].

resp text: <!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-la
Thought 2: Thought 2: Stuart Bird에 대한 정확한 정보가 없으니, 유사한 이름이나 관련 정보를 찾기 위해 Stuart Bird가 감독한 미국 액션 범죄 스릴러 영화를 검색해보는 것이 좋겠다.

Action 2: Search[Stuart Bird film]
Observation 2: Could not find Stuart Bird film. Similar: ['White Bird (film)', 'Stuart Little', 'Stuart Little (franchise)', 'Persu

KeyboardInterrupt: 